# 04 historial bancario y ensembles



## 5. Historial bancario: nuevas variables para predecir pagos

Después de las primeras pruebas incorporamos `comportamiento_bancario.csv`. Para cada solicitud reunimos los registros anteriores disponibles y calculamos indicadores de saldo y atrasos. La fecha del registro importa: no debemos utilizar información que todavía no existía cuando se solicitó el préstamo. Con estas variables evaluamos si el comportamiento bancario mejoraba la selección.

In [ ]:

columnas = [
    "Monto_Solicitado",
    "Tasa_Interes",
    "Ingreso_Mensual",
    "Ratio_Monto_Ingreso"
]

print("ENTRENAMIENTO")
display(train[columnas].describe())

print("TEST DE KAGGLE")
display(test[columnas].describe())


In [ ]:
prob_test = modelo_final.predict_proba(X_test)[:, 1]

In [ ]:

print("VALIDACIÓN")
print(pd.Series(probabilidades).describe())

print("TEST")
print(pd.Series(prob_test).describe())


In [ ]:

analisis = pd.DataFrame({
    "Prob_Pago": probabilidades,
    "Estado_Real": y_val.to_numpy()
})

analisis["Grupo"] = pd.qcut(
    analisis["Prob_Pago"],
    q=5,
    duplicates="drop"
)

resultado = analisis.groupby(
    "Grupo",
    observed=True
).agg(
    Probabilidad_Promedio=("Prob_Pago", "mean"),
    Porcentaje_Real_Pago=("Estado_Real", "mean"),
    Cantidad=("Estado_Real", "count")
)

display(resultado)


In [ ]:

# Recuperamos las solicitudes de validación
analisis_val = train.loc[X_val.index, [
    "Monto_Solicitado",
    "Tasa_Interes",
    "Estado_Final"
]].copy()

# Agregamos las probabilidades estimadas
analisis_val["Prob_Pago"] = probabilidades

# Seleccionamos préstamos de hasta USD 3.000
prestamos_pequenos = analisis_val[
    analisis_val["Monto_Solicitado"] <= 3000
]

print("Cantidad de préstamos pequeños:",
      len(prestamos_pequenos))

print("Probabilidad media estimada:",
      prestamos_pequenos["Prob_Pago"].mean())

print("Porcentaje real de pago:",
      prestamos_pequenos["Estado_Final"].mean())

print("Tasa promedio:",
      prestamos_pequenos["Tasa_Interes"].mean())


In [ ]:
# cargamos el historial bancario de los usuarios
bancario = pd.read_csv("comportamiento_bancario.csv")
display(bancario.head())


In [ ]:

print("Dimensiones:")
print(bancario.shape)

print("\nTipos de datos:")
bancario.info()

print("\nPrimeros registros:")
display(bancario.head(10))

print("\nValores nulos:")
print(bancario.isnull().sum())

print("\nMeses registrados por usuario:")
print(bancario.groupby("ID_Usuario").size().describe())


In [ ]:

print("Fechas de solicitudes históricas:")
print(train["Fecha_Solicitud"].min())
print(train["Fecha_Solicitud"].max())

print("\nFechas de solicitudes de test:")
print(test["Fecha_Solicitud"].min())
print(test["Fecha_Solicitud"].max())

print("\nMeses del comportamiento bancario:")
print(bancario["Mes_Referencia"].min())
print(bancario["Mes_Referencia"].max())

print("\nUsuarios de train con historial bancario:")
print(
    train["ID_Usuario"].isin(
        bancario["ID_Usuario"]
    ).mean() * 100
)

print("\nUsuarios de test con historial bancario:")
print(
    test["ID_Usuario"].isin(
        bancario["ID_Usuario"]
    ).mean() * 100
)


In [ ]:

# Convertimos las fechas a un formato temporal
train["Fecha_Solicitud"] = pd.to_datetime(
    train["Fecha_Solicitud"]
)

test["Fecha_Solicitud"] = pd.to_datetime(
    test["Fecha_Solicitud"]
)

bancario["Mes_Referencia"] = pd.to_datetime(
    bancario["Mes_Referencia"]
)

# Comprobamos los tipos
print(train["Fecha_Solicitud"].dtype)
print(test["Fecha_Solicitud"].dtype)
print(bancario["Mes_Referencia"].dtype)


In [ ]:
pd.Timestamp("2025-02-01") < pd.Timestamp("2025-05-01")

In [ ]:

historial = train[
    ["ID_Solicitud", "ID_Usuario", "Fecha_Solicitud"]
].merge(
    bancario,
    on="ID_Usuario",
    how="left"
)

print("Registros después del cruce:", historial.shape)
display(historial.head())


In [ ]:

historial = historial[
    historial["Mes_Referencia"] < historial["Fecha_Solicitud"]
].copy()

print("Registros históricos válidos:", historial.shape)

print("Meses disponibles por solicitud:")
display(
    historial.groupby("ID_Solicitud")
    .size()
    .describe()
)


In [ ]:

features_bancarias = historial.groupby(
    "ID_Solicitud"
).agg(
    Saldo_Promedio=(
        "Saldo_Promedio_Mensual", "mean"
    ),
    Saldo_Minimo=(
        "Saldo_Promedio_Mensual", "min"
    ),
    Atraso_Maximo=(
        "Dias_Atraso_Otras_Deudas", "max"
    ),
    Atraso_Promedio=(
        "Dias_Atraso_Otras_Deudas", "mean"
    ),
    Meses_Observados=(
        "Mes_Referencia", "count"
    )
).reset_index()

display(features_bancarias.head())

print("Dimensiones:", features_bancarias.shape)


In [ ]:

train_bancario = train.merge(
    features_bancarias,
    on="ID_Solicitud",
    how="left",
    validate="one_to_one"
)

print("Dimensiones:", train_bancario.shape)

print("Valores nulos en nuevas características:")
print(train_bancario[
    [
        "Saldo_Promedio",
        "Saldo_Minimo",
        "Atraso_Maximo",
        "Atraso_Promedio",
        "Meses_Observados"
    ]
].isnull().sum())


In [ ]:

columnas_modelo = [
    "Monto_Solicitado",
    "Tasa_Interes",
    "Ingreso_Mensual",
    "Ratio_Monto_Ingreso",
    "Saldo_Promedio",
    "Saldo_Minimo",
    "Atraso_Maximo",
    "Atraso_Promedio",
    "Meses_Observados"
]

X_bancario = train_bancario[columnas_modelo]

y_bancario = train_bancario["Estado_Final"]

print("X:", X_bancario.shape)
print("y:", y_bancario.shape)


In [ ]:

X_train_bancario = X_bancario.loc[X_train.index]
X_val_bancario = X_bancario.loc[X_val.index]

print("Train:", X_train_bancario.shape)
print("Validation:", X_val_bancario.shape)


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

modelo_bancario = Pipeline([
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

modelo_bancario.fit(
    X_train_bancario,
    y_train
)


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

predicciones_bancarias = modelo_bancario.predict(
    X_val_bancario
)

probabilidades_bancarias = modelo_bancario.predict_proba(
    X_val_bancario
)[:, 1]

print(
    "Accuracy:",
    accuracy_score(y_val, predicciones_bancarias)
)

print("\nMatriz de confusión:")
print(confusion_matrix(y_val, predicciones_bancarias))

print("\nReporte:")
print(classification_report(y_val, predicciones_bancarias))


In [ ]:

# Recuperamos los préstamos de validación
cartera_val_bancaria = train_bancario.loc[
    X_val.index,
    [
        "ID_Solicitud",
        "Monto_Solicitado",
        "Tasa_Interes",
        "Estado_Final"
    ]
].copy()

# Agregamos las probabilidades estimadas
cartera_val_bancaria["Prob_Pago"] = probabilidades_bancarias

cartera_val_bancaria["Prob_Default"] = (
    1 - cartera_val_bancaria["Prob_Pago"]
)

# Calculamos el valor esperado
cartera_val_bancaria["EV"] = (
    cartera_val_bancaria["Prob_Pago"]
    * cartera_val_bancaria["Monto_Solicitado"]
    * cartera_val_bancaria["Tasa_Interes"]
    -
    cartera_val_bancaria["Prob_Default"]
    * cartera_val_bancaria["Monto_Solicitado"]
)

display(cartera_val_bancaria.head())


In [ ]:

cartera_val_bancaria = cartera_val_bancaria.sort_values(
    "EV",
    ascending=False
)

presupuesto = 2_000_000
capital_usado_bancario = 0

cartera_val_bancaria["Aprobar"] = 0

for indice, fila in cartera_val_bancaria.iterrows():

    if fila["EV"] <= 0:
        continue

    if (
        capital_usado_bancario + fila["Monto_Solicitado"]
        <= presupuesto
    ):
        cartera_val_bancaria.loc[indice, "Aprobar"] = 1

        capital_usado_bancario += fila["Monto_Solicitado"]

print("Capital utilizado:", capital_usado_bancario)
print(
    "Préstamos aprobados:",
    cartera_val_bancaria["Aprobar"].sum()
)


In [ ]:

def ganancia_real_bancaria(fila):

    if fila["Aprobar"] == 0:
        return 0

    if fila["Estado_Final"] == 1:
        return (
            fila["Monto_Solicitado"]
            * fila["Tasa_Interes"]
        )

    return -fila["Monto_Solicitado"]


cartera_val_bancaria["Ganancia_Real"] = (
    cartera_val_bancaria.apply(
        ganancia_real_bancaria,
        axis=1
    )
)

ganancia_bancaria = (
    cartera_val_bancaria["Ganancia_Real"].sum()
)

print("Ganancia real simulada:", ganancia_bancaria)
print("Ganancia Baseline 1:", 347031.15)
print("Diferencia:", ganancia_bancaria - 347031.15)


In [ ]:

# armamos una tabla para analizar las probabilidades
analisis_bancario = train_bancario.loc[
    X_val.index,
    ["Monto_Solicitado", "Estado_Final"]
].copy()

analisis_bancario["Prob_Pago"] = probabilidades_bancarias

# nos quedamos con los prestamos de hasta 3000
pequenos_bancario = analisis_bancario[
    analisis_bancario["Monto_Solicitado"] <= 3000
]

# comparamos lo que predijo el modelo con lo que paso realmente
print("Cantidad:", len(pequenos_bancario))
print("Probabilidad estimada:", pequenos_bancario["Prob_Pago"].mean())
print("Porcentaje real de pago:", pequenos_bancario["Estado_Final"].mean())


In [ ]:

# separamos las solicitudes por fecha
entrenamiento_temporal = (
    train_bancario["Fecha_Solicitud"] < "2025-10-01"
)

validacion_temporal = (
    train_bancario["Fecha_Solicitud"] >= "2025-10-01"
)

# usamos las mismas 9 variables del modelo bancario
X_train_temp = X_bancario.loc[entrenamiento_temporal]
X_val_temp = X_bancario.loc[validacion_temporal]

y_train_temp = y_bancario.loc[entrenamiento_temporal]
y_val_temp = y_bancario.loc[validacion_temporal]

print("Entrenamiento:", X_train_temp.shape)
print("Validacion:", X_val_temp.shape)

print("Porcentaje de pago en entrenamiento:", y_train_temp.mean())
print("Porcentaje de pago en validacion:", y_val_temp.mean())


In [ ]:

# entrenamos la regresion logistica con los meses anteriores
modelo_bancario_temp = Pipeline([
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

modelo_bancario_temp.fit(X_train_temp, y_train_temp)

# evaluamos sobre los ultimos tres meses de 2025
pred_temp = modelo_bancario_temp.predict(X_val_temp)
prob_temp = modelo_bancario_temp.predict_proba(X_val_temp)[:, 1]

print("Accuracy:", accuracy_score(y_val_temp, pred_temp))
print("Matriz de confusion:")
print(confusion_matrix(y_val_temp, pred_temp))


In [ ]:

# calcula la ganancia de una cartera respetando el presupuesto
def evaluar_cartera(datos, probabilidades, presupuesto=2_000_000):
    cartera = datos[
        ["Monto_Solicitado", "Tasa_Interes", "Estado_Final"]
    ].copy()

    cartera["Prob_Pago"] = probabilidades

    # calculamos el valor esperado y ordenamos los prestamos
    cartera["EV"] = (
        cartera["Monto_Solicitado"] *
        (
            cartera["Prob_Pago"] * cartera["Tasa_Interes"]
            - (1 - cartera["Prob_Pago"])
        )
    )

    cartera = cartera.sort_values("EV", ascending=False)
    cartera["Aprobar"] = 0

    # seleccionamos los prestamos sin superar los 2 millones
    capital = 0

    for indice, fila in cartera.iterrows():
        monto = fila["Monto_Solicitado"]

        if fila["EV"] > 0 and capital + monto <= presupuesto:
            cartera.loc[indice, "Aprobar"] = 1
            capital += monto

    # calculamos lo que realmente habriamos ganado o perdido
    cartera["Ganancia_Real"] = 0.0

    aprobados = cartera["Aprobar"] == 1
    pagados = cartera["Estado_Final"] == 1

    cartera.loc[aprobados & pagados, "Ganancia_Real"] = (
        cartera["Monto_Solicitado"] * cartera["Tasa_Interes"]
    )

    cartera.loc[aprobados & ~pagados, "Ganancia_Real"] = (
        -cartera["Monto_Solicitado"]
    )

    print("Prestamos aprobados:", cartera["Aprobar"].sum())
    print("Capital utilizado:", capital)
    print("Ganancia real:", cartera["Ganancia_Real"].sum())

    return cartera


In [ ]:

# calculamos la ganancia del modelo con validacion temporal
cartera_temporal = evaluar_cartera(
    train_bancario.loc[X_val_temp.index],
    prob_temp
)


In [ ]:

# usamos solo las 4 variables originales
columnas_base = [
    "Monto_Solicitado",
    "Tasa_Interes",
    "Ingreso_Mensual",
    "Ratio_Monto_Ingreso"
]

X_base_temp_train = train.loc[
    entrenamiento_temporal,
    columnas_base
]

X_base_temp_val = train.loc[
    validacion_temporal,
    columnas_base
]

# entrenamos el modelo base con las mismas fechas
modelo_base_temp = Pipeline([
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

modelo_base_temp.fit(
    X_base_temp_train,
    y_train_temp
)

# obtenemos las probabilidades para validacion
prob_base_temp = modelo_base_temp.predict_proba(
    X_base_temp_val
)[:, 1]

print(
    "Accuracy base temporal:",
    accuracy_score(
        y_val_temp,
        modelo_base_temp.predict(X_base_temp_val)
    )
)


In [ ]:

# calculamos la ganancia del modelo sin historial bancario
cartera_base_temporal = evaluar_cartera(
    train.loc[X_base_temp_val.index],
    prob_base_temp
)


In [ ]:

# juntamos las solicitudes de test con su historial bancario
historial_test = test[
    ["ID_Solicitud", "ID_Usuario", "Fecha_Solicitud"]
].merge(
    bancario,
    on="ID_Usuario",
    how="left"
)

# dejamos solo los meses anteriores a cada solicitud
historial_test = historial_test[
    historial_test["Mes_Referencia"] < historial_test["Fecha_Solicitud"]
].copy()

# calculamos las mismas caracteristicas que usamos en entrenamiento
features_bancarias_test = historial_test.groupby(
    "ID_Solicitud"
).agg(
    Saldo_Promedio=("Saldo_Promedio_Mensual", "mean"),
    Saldo_Minimo=("Saldo_Promedio_Mensual", "min"),
    Atraso_Maximo=("Dias_Atraso_Otras_Deudas", "max"),
    Atraso_Promedio=("Dias_Atraso_Otras_Deudas", "mean"),
    Meses_Observados=("Mes_Referencia", "count")
).reset_index()

# agregamos las caracteristicas a las solicitudes de test
test_bancario = test.merge(
    features_bancarias_test,
    on="ID_Solicitud",
    how="left",
    validate="one_to_one"
)

print("Test:", test_bancario.shape)
print("Nulos:", test_bancario[columnas_modelo].isnull().sum().sum())


In [ ]:

# entrenamos el modelo con todas las solicitudes historicas
modelo_final_v3 = Pipeline([
    ("scaler", StandardScaler()),
    ("modelo", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

modelo_final_v3.fit(
    train_bancario[columnas_modelo],
    train_bancario["Estado_Final"]
)

# obtenemos la probabilidad de pago de cada solicitud nueva
prob_test_v3 = modelo_final_v3.predict_proba(
    test_bancario[columnas_modelo]
)[:, 1]

print("Probabilidades calculadas:", len(prob_test_v3))
print(pd.Series(prob_test_v3).describe())


In [ ]:

# armamos la cartera de test con las probabilidades
cartera_test_v3 = test_bancario[
    ["ID_Solicitud", "Monto_Solicitado", "Tasa_Interes"]
].copy()

cartera_test_v3["Prob_Pago"] = prob_test_v3

# calculamos la ganancia esperada de cada prestamo
cartera_test_v3["EV"] = (
    cartera_test_v3["Monto_Solicitado"] *
    (
        cartera_test_v3["Prob_Pago"] * cartera_test_v3["Tasa_Interes"]
        - (1 - cartera_test_v3["Prob_Pago"])
    )
)

# ordenamos por ganancia esperada
cartera_test_v3 = cartera_test_v3.sort_values(
    "EV",
    ascending=False
)

# elegimos los prestamos respetando el limite de capital
presupuesto_v3 = 2_000_000
capital_v3 = 0

cartera_test_v3["Aprobar"] = 0

for indice, fila in cartera_test_v3.iterrows():
    monto = fila["Monto_Solicitado"]

    if fila["EV"] > 0 and capital_v3 + monto <= presupuesto_v3:
        cartera_test_v3.loc[indice, "Aprobar"] = 1
        capital_v3 += monto

print("Prestamos aprobados:", cartera_test_v3["Aprobar"].sum())
print("Capital utilizado:", capital_v3)
print("Presupuesto restante:", presupuesto_v3 - capital_v3)


In [ ]:

# recuperamos el orden original de las solicitudes
submission_v3 = test[["ID_Solicitud"]].merge(
    cartera_test_v3[["ID_Solicitud", "Aprobar"]],
    on="ID_Solicitud",
    how="left",
    validate="one_to_one"
)

# comprobamos que no falten decisiones y que no superemos el presupuesto
assert len(submission_v3) == len(test)
assert submission_v3["Aprobar"].notna().all()
assert submission_v3["Aprobar"].isin([0, 1]).all()
assert capital_v3 <= 2_000_000

# dejamos las columnas en el mismo formato que submission v1
submission_v3 = submission_v3.rename(
    columns={"Aprobar": "Estado_Final"}
)

submission_v3.to_csv("submission_v3.csv", index=False)

print("Archivo generado: submission_v3.csv")
display(submission_v3.head())


In [ ]:

# descargamos el archivo para subirlo a kaggle

print("Archivo disponible para descargar:", "submission_v3.csv")

In [ ]:

# cambiamos el nombre de la columna al que pide kaggle
submission_v3 = submission_v3.rename(
    columns={"Estado_Final": "Decision_Aprobacion"}
)

# guardamos el archivo corregido
submission_v3.to_csv("submission_v3.csv", index=False)

# descargamos el nuevo archivo

print("Archivo disponible para descargar:", "submission_v3.csv")
print(submission_v3.head())


### Modelo 6 — Random Forest con información bancaria

Probamos Random Forest con las variables bancarias para ver si un conjunto de árboles detectaba relaciones que la regresión logística no estaba aprovechando. Evaluamos probabilidades, ganancia y uso del presupuesto con el mismo criterio financiero. No lo elegimos como solución final porque las comparaciones posteriores no mostraron una ventaja sostenida frente a la regresión.

In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# entrenamos random forest con los datos bancarios
modelo_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

modelo_rf.fit(X_train_temp, y_train_temp)

# hacemos las predicciones sobre la validacion temporal
pred_rf = modelo_rf.predict(X_val_temp)
prob_rf = modelo_rf.predict_proba(X_val_temp)[:, 1]

# vemos las metricas del modelo
print("Accuracy:", accuracy_score(y_val_temp, pred_rf))
print("\nMatriz de confusion:")
print(confusion_matrix(y_val_temp, pred_rf))
print("\nReporte:")
print(classification_report(y_val_temp, pred_rf))


In [ ]:

# evaluamos la ganancia usando el presupuesto de 2 millones
print("MODELO 6 - RANDOM FOREST")

cartera_rf = evaluar_cartera(
    train_bancario.loc[X_val_temp.index],
    prob_rf
)

# comparamos la ganancia con el modelo anterior
ganancia_rf = cartera_rf["Ganancia_Real"].sum()
ganancia_logistica = 608210.86

print("\nGanancia Random Forest:", round(ganancia_rf, 2))
print("Ganancia Regresion Logistica:", ganancia_logistica)
print("Diferencia:", round(ganancia_rf - ganancia_logistica, 2))


In [ ]:

# ordenamos las variables segun su importancia en random forest
importancias_rf = pd.DataFrame({
    "Caracteristica": columnas_modelo,
    "Importancia": modelo_rf.feature_importances_
}).sort_values("Importancia", ascending=False)

display(importancias_rf)


In [ ]:

# entrenamos el modelo definitivo con todas las solicitudes historicas
modelo_rf_final = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

modelo_rf_final.fit(
    train_bancario[columnas_modelo],
    train_bancario["Estado_Final"]
)

# obtenemos las probabilidades de pago de kaggle
prob_rf_test = modelo_rf_final.predict_proba(
    test_bancario[columnas_modelo]
)[:, 1]

print("Solicitudes de test:", len(prob_rf_test))
print(pd.Series(prob_rf_test).describe())


In [ ]:

# armamos la cartera de test
cartera_rf_test = test_bancario[
    ["ID_Solicitud", "Monto_Solicitado", "Tasa_Interes"]
].copy()

cartera_rf_test["Prob_Pago"] = prob_rf_test

# calculamos el valor esperado de cada prestamo
cartera_rf_test["EV"] = (
    cartera_rf_test["Monto_Solicitado"] *
    (
        cartera_rf_test["Prob_Pago"] * cartera_rf_test["Tasa_Interes"]
        - (1 - cartera_rf_test["Prob_Pago"])
    )
)

# ordenamos los prestamos por ganancia esperada
cartera_rf_test = cartera_rf_test.sort_values(
    "EV",
    ascending=False
)

# aprobamos prestamos sin superar el presupuesto
presupuesto_rf = 2_000_000
capital_rf = 0

cartera_rf_test["Decision_Aprobacion"] = 0

for indice, fila in cartera_rf_test.iterrows():
    monto = fila["Monto_Solicitado"]

    if fila["EV"] > 0 and capital_rf + monto <= presupuesto_rf:
        cartera_rf_test.loc[indice, "Decision_Aprobacion"] = 1
        capital_rf += monto

print("Prestamos aprobados:", cartera_rf_test["Decision_Aprobacion"].sum())
print("Capital utilizado:", capital_rf)
print("Presupuesto restante:", presupuesto_rf - capital_rf)


In [ ]:

# recuperamos el orden original de las solicitudes
submission_v4 = test[["ID_Solicitud"]].merge(
    cartera_rf_test[["ID_Solicitud", "Decision_Aprobacion"]],
    on="ID_Solicitud",
    how="left",
    validate="one_to_one"
)

# comprobamos que el archivo tenga todas las decisiones
assert len(submission_v4) == len(test)
assert submission_v4["Decision_Aprobacion"].notna().all()
assert submission_v4["Decision_Aprobacion"].isin([0, 1]).all()

# comprobamos el capital total aprobado
monto_total_v4 = (
    test_bancario[["ID_Solicitud", "Monto_Solicitado"]]
    .merge(submission_v4, on="ID_Solicitud")
    .assign(
        Monto_Aprobado=lambda df:
            df["Monto_Solicitado"] * df["Decision_Aprobacion"]
    )["Monto_Aprobado"].sum()
)

assert monto_total_v4 <= 2_000_000

# guardamos y descargamos la submission
submission_v4.to_csv("submission_v4.csv", index=False)

print("Capital aprobado:", monto_total_v4)
print("Archivo generado: submission_v4.csv")

print("Archivo disponible para descargar:", "submission_v4.csv")

### Modelo 7 — Gradient Boosting

Probamos Gradient Boosting sobre las características disponibles y revisamos cuáles pesaban más en las predicciones. Evaluamos su ganancia en validación antes de considerar una submission. No encontramos una mejora suficientemente clara como para reemplazar la regresión logística que veníamos utilizando.

In [ ]:

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# entrenamos gradient boosting con las variables bancarias
modelo_gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    min_samples_leaf=10,
    random_state=42
)

modelo_gb.fit(X_train_temp, y_train_temp)

# calculamos las predicciones en validacion temporal
pred_gb = modelo_gb.predict(X_val_temp)
prob_gb = modelo_gb.predict_proba(X_val_temp)[:, 1]

# mostramos las metricas
print("Accuracy:", accuracy_score(y_val_temp, pred_gb))
print("\nMatriz de confusion:")
print(confusion_matrix(y_val_temp, pred_gb))
print("\nReporte:")
print(classification_report(y_val_temp, pred_gb))


In [ ]:

# calculamos la ganancia de gradient boosting
print("MODELO 7 - GRADIENT BOOSTING")

cartera_gb = evaluar_cartera(
    train_bancario.loc[X_val_temp.index],
    prob_gb
)

# comparamos los tres modelos
ganancia_gb = cartera_gb["Ganancia_Real"].sum()

resultados_modelos = pd.DataFrame({
    "Modelo": [
        "Regresion Logistica",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Ganancia": [
        608210.86,
        511974.89,
        ganancia_gb
    ]
})

display(resultados_modelos.sort_values("Ganancia", ascending=False))


In [ ]:

# vemos que variables utiliza mas gradient boosting
importancias_gb = pd.DataFrame({
    "Caracteristica": columnas_modelo,
    "Importancia": modelo_gb.feature_importances_
}).sort_values("Importancia", ascending=False)

display(importancias_gb)


In [ ]:

# preparamos la submission solo si supera al modelo 5
if ganancia_gb > 608210.86:

    # entrenamos gradient boosting con todos los datos
    modelo_gb_final = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        min_samples_leaf=10,
        random_state=42
    )

    modelo_gb_final.fit(
        train_bancario[columnas_modelo],
        train_bancario["Estado_Final"]
    )

    # calculamos las probabilidades del test
    prob_gb_test = modelo_gb_final.predict_proba(
        test_bancario[columnas_modelo]
    )[:, 1]

    # armamos y ordenamos la cartera segun el valor esperado
    cartera_gb_test = test_bancario[
        ["ID_Solicitud", "Monto_Solicitado", "Tasa_Interes"]
    ].copy()

    cartera_gb_test["EV"] = (
        cartera_gb_test["Monto_Solicitado"] *
        (
            prob_gb_test * cartera_gb_test["Tasa_Interes"]
            - (1 - prob_gb_test)
        )
    )

    cartera_gb_test = cartera_gb_test.sort_values(
        "EV", ascending=False
    )

    # seleccionamos los prestamos sin superar el presupuesto
    cartera_gb_test["Decision_Aprobacion"] = 0
    capital_gb = 0

    for indice, fila in cartera_gb_test.iterrows():
        monto = fila["Monto_Solicitado"]

        if fila["EV"] > 0 and capital_gb + monto <= 2_000_000:
            cartera_gb_test.loc[indice, "Decision_Aprobacion"] = 1
            capital_gb += monto

    # guardamos las decisiones en el orden original
    submission_v4 = test[["ID_Solicitud"]].merge(
        cartera_gb_test[["ID_Solicitud", "Decision_Aprobacion"]],
        on="ID_Solicitud",
        how="left",
        validate="one_to_one"
    )

    # comprobamos el formato y el presupuesto
    assert len(submission_v4) == len(test)
    assert submission_v4["Decision_Aprobacion"].notna().all()
    assert submission_v4["Decision_Aprobacion"].isin([0, 1]).all()
    assert capital_gb <= 2_000_000

    submission_v4.to_csv("submission_v4.csv", index=False)

    print("Prestamos aprobados:", submission_v4["Decision_Aprobacion"].sum())
    print("Capital utilizado:", capital_gb)

    print("Archivo disponible para descargar:", "submission_v4.csv")
else:
    print("Gradient Boosting no supero al Modelo 5.")
    print("Por ahora no generamos otra submission.")


### Modelo 8 — Formas de ordenar los préstamos

Con las mismas probabilidades comparamos criterios de selección: ganancia esperada total, rentabilidad por dólar y una combinación de ambos. El objetivo era aprovechar mejor los USD 2 millones disponibles. A partir de estas pruebas seguimos prestando especial atención a la ganancia por dólar, porque el monto de cada préstamo también consume presupuesto.

In [ ]:

# compara distintas formas de ordenar los prestamos
def evaluar_estrategia(datos, probabilidades, estrategia="ev"):
    cartera = datos[
        ["Monto_Solicitado", "Tasa_Interes", "Estado_Final"]
    ].copy()

    cartera["Prob_Pago"] = probabilidades

    # calculamos el valor esperado de cada prestamo
    cartera["EV"] = (
        cartera["Monto_Solicitado"] *
        (
            cartera["Prob_Pago"] * cartera["Tasa_Interes"]
            - (1 - cartera["Prob_Pago"])
        )
    )

    cartera["EV_Por_Dolar"] = (
        cartera["EV"] / cartera["Monto_Solicitado"]
    )

    # elegimos el criterio para ordenar los prestamos
    if estrategia == "ev":
        cartera["Puntaje"] = cartera["EV"]

    elif estrategia == "rentabilidad":
        cartera["Puntaje"] = cartera["EV_Por_Dolar"]

    elif estrategia == "mixto":
        cartera["Puntaje"] = (
            cartera["EV_Por_Dolar"] *
            cartera["Monto_Solicitado"] ** 0.5
        )

    else:
        raise ValueError("Estrategia no valida")

    cartera = cartera.sort_values("Puntaje", ascending=False)
    cartera["Aprobar"] = 0

    # aprobamos los prestamos hasta completar el presupuesto
    capital = 0

    for indice, fila in cartera.iterrows():
        monto = fila["Monto_Solicitado"]

        if fila["EV"] > 0 and capital + monto <= 2_000_000:
            cartera.loc[indice, "Aprobar"] = 1
            capital += monto

    # calculamos la ganancia real de la cartera seleccionada
    aprobados = cartera["Aprobar"] == 1
    pagados = cartera["Estado_Final"] == 1

    cartera["Ganancia_Real"] = 0.0

    cartera.loc[aprobados & pagados, "Ganancia_Real"] = (
        cartera["Monto_Solicitado"] *
        cartera["Tasa_Interes"]
    )

    cartera.loc[aprobados & ~pagados, "Ganancia_Real"] = (
        -cartera["Monto_Solicitado"]
    )

    return {
        "Estrategia": estrategia,
        "Aprobados": int(cartera["Aprobar"].sum()),
        "Capital": capital,
        "Ganancia": cartera["Ganancia_Real"].sum()
    }


In [ ]:

# probamos las tres estrategias con el modelo bancario
resultados_estrategias = []

for estrategia in ["ev", "rentabilidad", "mixto"]:
    resultado = evaluar_estrategia(
        train_bancario.loc[X_val_temp.index],
        prob_temp,
        estrategia
    )

    resultados_estrategias.append(resultado)

# mostramos los resultados ordenados por ganancia
comparacion_estrategias = pd.DataFrame(resultados_estrategias)

display(
    comparacion_estrategias.sort_values(
        "Ganancia", ascending=False
    )
)
